**Indexing: Creating a Chroma Vectorstore**

In [1]:
%load_ext dotenv
%dotenv

In [2]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters.markdown import MarkdownHeaderTextSplitter
from langchain_text_splitters.character import CharacterTextSplitter
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
import numpy as np

In [3]:
# Load Text from Document
loader_docx = Docx2txtLoader("RAGFiles/Introduction_to_Data_and_Data_Science.docx")
pages = loader_docx.load()

# Markdown Header to Split Text into Lectures
md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on = [("#","Course Title"),
                           ("##","Lecture Title")]
    )

pages_md_split = md_splitter.split_text(pages[0].page_content)

# Removed Newline Characterss Cluttering Text
for i in range(len(pages_md_split)):
    pages_md_split[i].page_content = ' '.join(pages_md_split[i].page_content.split())

# Split Text within Lectures
char_splitter = CharacterTextSplitter(
    separator = ".",
    chunk_size = 500,
    chunk_overlap = 50
)

# Split into Chunks 
pages_char_split = char_splitter.split_documents(pages_md_split)

# Embedding Model Class (Assigning Similarity Scores)
embedding = OpenAIEmbeddings(
    model="text-embedding-3-small",
    chunk_size=1,          # <= key: send 1 text at a time
    request_timeout=180,   # longer timeout
    max_retries=12,        # retry on drops
)

In [4]:
# Specifies what documents to store, how to embed, and where to host (locally)
vectorstore = Chroma.from_documents(documents = pages_char_split, embedding = embedding, persist_directory = "./intro-to-ds-lectures")

In [5]:
# Needs the same embedding function as the embedding can change, or, new inputs could be adde
vectorstore_from_directory = Chroma(persist_directory = "./intro-to-ds-lectures", embedding_function=embedding)

OPENAI_API_KEY set? True
